# Imports

In [2]:
# --- Imports ---
import os
import sys

import numpy as np
import pandas as pd
from tqdm import tqdm

# --- Add Paths ---
# Go up TWO levels (project root directory)
project_root = os.path.dirname(os.path.dirname(os.getcwd()))

# Append the new path to sys.path
if project_root not in sys.path:
    sys.path.append(project_root)
    print("Project root added to sys.path")
else:
    print("Project root already in sys.path")

Project root added to sys.path


# User Selects Atoms

Obtain `.gro` file data as `df`.

In [29]:
# get dataframe
from utils import gro_processing

# Get data
df, title, num_atoms, box_dimensions = gro_processing.read_gro("../../data/npt-HK4.gro")
df_gro = gro_processing.dataframe_gro(df, box_dimensions[0], oxygen_midpoints=False)

# df.head()
# df_gro.head()
# title
# num_atoms
# box_dimensions

Some basic parameters for a single residue (molecule).

In [ ]:
# parameters
res_id = 1
num_atoms_per_mol = df_gro.loc[(res_id,),:].shape[0]
num_res = df_gro.index.levels[0][-1]

print(f'{num_atoms_per_mol=}')
print(f'{num_res=}')

User selects indices (atoms) of molecule to be considered _electron clump cluster_.

In [25]:
# user selects atoms within a single molecule
from itertools import chain

def extend_input(user_input):
    if '-' in user_input:
        start, end = user_input.split('-')
        return [*range(int(start), int(end) + 1)]
    else:
        return [int(user_input)]

# user chooses atoms within molecule
print("Insert atom id for one molecule.")
print("Use the following format: '20-30; 30; 20; 40-60' (for range use '-', for multi-input split using ';' ")
# user_input = input("Enter atom id:")
user_input = "15-20; 30; 31-33"  # for testing purposes
print(f"\nYou entered: '{user_input}'")

# extract atom indices that user selected
try: 
    indices = [extend_input(i) for i in user_input.split(";")]
    indices = np.unique(list(chain.from_iterable(indices)))
except: 
    print("Invalid input format. Please use the specified format.")

indices


Insert atom id for one molecule.
Use the following format: '20-30; 30; 20; 40-60' (for range use '-', for multi-input split using ';' 

You entered: '15-20; 30; 31-33'


array([15, 16, 17, 18, 19, 20, 30, 31, 32, 33])

Identify _electron clump cluster_ for all molecules in `.gro` file (assuming all molecules are the same).

In [27]:
# filters all molecules based on user selected atoms
# indices = indices + (num_atoms_per_mol * (res_id - 1)) # filter single atom
indices_all = [indices]

for i in range(1, num_res):
    indices_all.append(indices + i * num_atoms_per_mol)
    
# flatten the list 
indices_all = np.concatenate(indices_all)
display(indices_all)

# extract data
mask = df_gro["atom_id"].isin(indices_all)
new_df = df_gro.loc[mask].copy()
# display(new_df)

array([    15,     16,     17, ..., 126031, 126032, 126033],
      shape=(15010,))

# Centroids

In [8]:
# Finding centroid from atoms in new_df
def minimum_image(dx, box_dimensions):
    k = np.rint(dx * (1 / box_dimensions)) # handles 3-D case correctly
    return dx - k * box_dimensions

def minimum_image_vector(p1, p2, box_dimensions):
    dx = p1 - p2
    return minimum_image(dx, box_dimensions)

def wrap_points(points, box_dimensions):
    return points % box_dimensions

In [9]:
# calculate centroid of single molecule
res_id = 1
temp_df = new_df.loc[(res_id,slice(None)),['x','y','z']]
# display(temp_df)
res_atoms = temp_df.to_numpy()
ref_atom = res_atoms[0]
rel_displacement = np.zeros_like(res_atoms)

for i in range(res_atoms.shape[0]):
    rel_displacement[i] = minimum_image_vector(res_atoms[i], ref_atom, box_dimensions)
# display(rel_displacement)

mean_displacement = np.mean(rel_displacement, axis=0)
centroids_unwrapped = ref_atom + mean_displacement
centroid = wrap_points(centroids_unwrapped, box_dimensions)

# centroid = ref_atom + np.mean(rel_displacement, axis=0)
print(centroid)


[1.5018 0.3441 0.3525]


In [10]:
# calculate centroids for all molecules

# chunk numpy arrays
e_centroids = np.zeros((num_res, 3))

for res_id in range(1, num_res + 1):
    # selects residue (molecule) one-by-one
    res_atoms = new_df.loc[(res_id,slice(None)),['x','y','z']].to_numpy()
    ref_atom = res_atoms[0]
    rel_displacement = np.zeros_like(res_atoms)  # rel displacement of all atoms from ref_atom
    for i in range(res_atoms.shape[0]):
        rel_displacement[i] = minimum_image_vector(res_atoms[i], ref_atom, box_dimensions)
    
    # calculate centroid of the molecule from origin [0,0,0]
    mean_displacement = np.mean(rel_displacement, axis=0)
    centroids_unwrapped = ref_atom + mean_displacement
    e_centroids[res_id-1] = wrap_points(centroids_unwrapped, box_dimensions)

# convert to dataframe
df_e_centroids = pd.DataFrame(e_centroids, columns=['x', 'y', 'z'], index=range(1, num_res + 1))

df_e_centroids.head()

,x,y,z
1,1.5018,0.344100,0.352500
2,0.3915,0.635900,0.031712
3,0.3374,10.928200,0.872900
4,0.5964,1.113500,0.247500
5,5.7596,0.019406,5.378100
